In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

path = "../simulacoes/financeiro/comparativo_metricas.csv"
df = pd.read_csv(path)

In [ ]:
df_long = df.melt(
    id_vars=["janela_previsao", "estatistica"], 
    var_name="configuracao", 
    value_name="valor"
)

In [ ]:
df_long[["tipo_dado", "modelo", "lookback"]] = df_long["configuracao"].str.split("__", expand=True)

In [ ]:
df_wide = df_long.pivot_table(
    index=["janela_previsao", "tipo_dado", "modelo", "lookback"],
    columns="estatistica",
    values="valor"
).reset_index()
df_wide.columns.name = None


df_export = df_wide[
    df_wide['mean_precision_positive'] >= .595
].sort_values(by='mean_precision_positive', ascending=False)


df_export.to_csv("dados_pivotados.csv", index=False)

In [ ]:
pct_prices = (df_export['tipo_dado'] == 'prices').mean() * 100
print(f"A. Percentual de 'prices' em tipo_dado: {pct_prices:.2f}%")
print("-" * 50)

# B. Distribuição percentual de look_back
print("B. Distribuição percentual de lookback:")
dist_lookback = df_export['lookback'].value_counts(normalize=True) * 100
print(dist_lookback.map("{:.2f}%".format))
print("-" * 50)

# C. Distribuição percentual de janela de previsão
print("C. Distribuição percentual de janela_previsao:")
dist_janela = df_export['janela_previsao'].value_counts(normalize=True) * 100
print(dist_janela.map("{:.2f}%".format))
print("-" * 50)

# D. Distribuição percentual de modelo
print("D. Distribuição percentual de modelo:")
dist_modelo = df_export['modelo'].value_counts(normalize=True) * 100
print(dist_modelo.map("{:.2f}%".format))
print("-" * 50)

In [ ]:
df_precision = df_long[df_long['estatistica'] == 'mean_precision_positive']

# 3. Configurar o estilo visual do seaborn
sns.set_theme(style="whitegrid")

# 4. Criar o diagrama de dispersão com subgráficos (facets) para 'tipo_dado'
g = sns.relplot(
    data=df_precision,
    x="janela_previsao",
    y="valor",
    hue="modelo",          # Cor do ponto mapeia o modelo
    style="lookback",      # Formato do ponto mapeia o lookback
    col="tipo_dado",       # SEPARAÇÃO: Uma coluna para cada tipo de dado!
    kind="scatter",
    s=100,                 # Tamanho dos pontos aumentado para melhor visibilidade
    alpha=0.8,             # Leve transparência para pontos sobrepostos
    palette="deep",        # Paleta de cores moderna e legível
    height=5,
    aspect=1.2
)

# 5. Ajustar títulos e rótulos dos eixos
g.set_axis_labels("Janela de Previsão", "Mean Precision Positive")
g.set_titles("Tipo de Dado: {col_name}")

# Ajustar título principal
g.figure.subplots_adjust(top=0.85)
g.figure.suptitle(
    "Visão Geral de Mean Precision Positive por Configuração", 
    fontsize=14, 
    weight='bold'
)

# 6. Salvar o gráfico como imagem
plt.savefig("distribuicao_precision.png", bbox_inches='tight', dpi=300)
